# Analisis Pola Harian Kualitas Udara (PM2.5)

## Deskripsi Penelitian

Notebook ini melakukan analisis pola harian (hourly pattern) konsentrasi PM2.5 untuk memahami variasi kualitas udara berdasarkan waktu. Analisis ini mencakup:

1. **Pola Harian Umum** - Variasi PM2.5 berdasarkan jam dalam sehari
2. **Perbandingan Hari Kerja vs Akhir Pekan** - Perbedaan perilaku polusi
3. **Statistik Deskriptif** - Rata-rata, standar deviasi, min, max per jam
4. **Visualisasi** - Grafik line chart dan heatmap untuk bukti penelitian

## Latar Belakang

Pola harian kualitas udara dipengaruhi oleh:
- Aktivitas manusia (transporasi, industri, domestik)
- Kondisi meteorologi (suhu, kelembaban, arah angin)
- Sirkulasi atmosfer (mixing height, stabilitas udara)

**Jam Puncak Polusi:**
- Pagi: 06:00 - 09:00 (aktivitas通勤, inversi suhu pagi)
- Sore: 17:00 - 19:00 (puncak lalu lintas sore)

**Jam Kualitas Baik:**
- Malam-Pagi dini: 00:00 - 05:00 (atmosfer stabil, emisi rendah)
- Siang: 12:00 - 15:00 (mixing layer tinggi, dispersi baik)

In [ ]:
# Import library yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Pengaturan visualisasi
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 10

print("Library berhasil diimport!")

## 1. Persiapan Data

Data diambil dari database Supabase atau menggunakan data sintetis yang menyerupai pola PM2.5 nyata.

In [ ]:
# Generate data sintetis yang menyerupai pola PM2.5 nyata
# Pola ini berdasarkan karakteristik data kualitas udara di Indonesia

np.random.seed(42)
n_days = 60  # 60 hari data
n_per_day = 24  # 1 data per jam

dates = pd.date_range('2025-01-01', periods=n_days * n_per_day, freq='H')

# Base pattern: lebih tinggi di jam sibuk
hours = dates.hour

# Pola harian PM2.5 (μg/m³):
# 00-05: rendah (15-18) - atmosfer stabil
# 06-09: naik tajam (25-35) - aktivitas pagi
# 10-14: tinggi (30-35) - puncak aktivitas
# 15-17: mulai turun (25-28)
# 18-21: naik lagi (28-32) - aktivitas sore
# 22-23: turun (20-22)

base_pm25 = np.where(
    (hours >= 0) & (hours < 6), 15,
    np.where(
        (hours >= 6) & (hours < 9), 28,
        np.where(
            (hours >= 9) & (hours < 12), 33,
            np.where(
                (hours >= 12) & (hours < 15), 30,
                np.where(
                    (hours >= 15) & (hours < 18), 26,
                    np.where(
                        (hours >= 18) & (hours < 21), 30,
                        20  # 21-23
                    )
                )
            )
        )
    )
)

# Variasi hari kerja vs akhir pekan
# Hari kerja (Senin-Jumat) = 1, Akhir pekan (Sabtu-Minggu) = 0
day_of_week = pd.Series(dates).dt.dayofweek.values
is_weekday = (day_of_week < 5).astype(int)

# Hari kerja memiliki PM2.5 ~20% lebih tinggi
pm25_base = base_pm25 * (1 + 0.2 * (is_weekday - 0.5))

# Noise acak
noise = np.random.normal(0, 4, len(dates))

# PM2.5 final (tidak boleh negatif)
pm25 = np.maximum(pm25_base + noise, 5)

# Buat DataFrame
df = pd.DataFrame({
    'timestamp': dates,
    'pm25': pm25,
    'hour': hours,
    'day_of_week': day_of_week,
    'is_weekday': is_weekday,
    'date': dates.date
})

# Label hari
df['day_type'] = df['is_weekday'].map({1: 'Hari Kerja', 0: 'Akhir Pekan'})

print(f"Dataset berhasil dibuat!")
print(f"Total data: {len(df)} records")
print(f"Periode: {df['timestamp'].min()} s/d {df['timestamp'].max()}")
print(f"\nStatistik PM2.5:")
print(df['pm25'].describe())

## 2. Analisis Statistik Deskriptif

Melihat statistik dasar data PM2.5:

In [ ]:
# Statistik deskriptif keseluruhan
print("=" * 60)
print("STATISTIK DESKRIPTIF PM2.5")
print("=" * 60)

print(f"\nJumlah observasi: {len(df)}")
print(f"Rata-rata: {df['pm25'].mean():.2f} μg/m³")
print(f"Median: {df['pm25'].median():.2f} μg/m³")
print(f"Std Dev: {df['pm25'].std():.2f} μg/m³")
print(f"Min: {df['pm25'].min():.2f} μg/m³")
print(f"Max: {df['pm25'].max():.2f} μg/m³")

# Statistik per jam
hourly_stats = df.groupby('hour')['pm25'].agg(['mean', 'std', 'min', 'max', 'count'])
hourly_stats.columns = ['Rata-rata', 'Std Dev', 'Min', 'Max', 'Jumlah']
print("\n" + "=" * 60)
print("STATISTIK PM2.5 PER JAM")
print("=" * 60)
print(hourly_stats.round(2).to_string())

## 3. Visualisasi Pola Harian

### 3.1 Line Chart Pola Harian PM2.5

In [ ]:
# Aggregate data per jam
hourly_avg = df.groupby('hour')['pm25'].mean()
hourly_avg_weekday = df[df['is_weekday'] == 1].groupby('hour')['pm25'].mean()
hourly_avg_weekend = df[df['is_weekday'] == 0].groupby('hour')['pm25'].mean()

# Buat visualisasi
fig, ax = plt.subplots(figsize=(14, 7))

# Plot garis untuk semua data
ax.plot(hourly_avg.index, hourly_avg.values, 
         color='#6366f1', linewidth=3, marker='o', markersize=8,
         label='Rata-rata Keseluruhan', zorder=5)

# Plot garis hari kerja
ax.plot(hourly_avg_weekday.index, hourly_avg_weekday.values,
         color='#3b82f6', linewidth=2.5, linestyle='-', marker='s', markersize=6,
         label='Hari Kerja (Senin-Jumat)', alpha=0.9)

# Plot garis akhir pekan
ax.plot(hourly_avg_weekend.index, hourly_avg_weekend.values,
         color='#f59e0b', linewidth=2.5, linestyle='--', marker='^', markersize=6,
         label='Akhir Pekan (Sabtu-Minggu)', alpha=0.9)

# Zona jam sibuk
ax.axvspan(6, 9, alpha=0.15, color='red', label='Jam Sibuk Pagi (06:00-09:00)')
ax.axvspan(17, 20, alpha=0.15, color='orange', label='Jam Sibuk Sore (17:00-20:00)')

# Garis horizontal kategori ISPU
ax.axhline(y=50, color='green', linestyle=':', alpha=0.7, label='Batas ISPU Baik (50)')
ax.axhline(y=100, color='blue', linestyle=':', alpha=0.7, label='Batas ISPU Sedang (100)')

# Formatting
ax.set_xlabel('Jam (WIB)', fontsize=12)
ax.set_ylabel('Konsentrasi PM2.5 (μg/m³)', fontsize=12)
ax.set_title('Pola Harian PM2.5\nPerbandingan Hari Kerja vs Akhir Pekan', fontsize=16, fontweight='bold')
ax.set_xticks(range(0, 24, 1))
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0, 50)
ax.legend(loc='upper left', framealpha=0.95, ncol=2)
ax.grid(True, alpha=0.3)

# Annotasi jam sibuk
ax.annotate('Puncak\nPagi', xy=(7, 33), xytext=(4, 42),
            fontsize=10, ha='center',
            arrowprops=dict(arrowstyle='->', color='gray'))

ax.annotate('Puncak\nSore', xy=(19, 34), xytext=(22, 42),
            fontsize=10, ha='center',
            arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.savefig('pola_harian_pm25.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nGambar disimpan: pola_harian_pm25.png")

### 3.2 Tabel Pola Harian (Data Numerik)

In [ ]:
# Tabel perbandingan pola harian
pola_table = pd.DataFrame({
    'Jam': [f'{h:02d}:00' for h in range(24)],
    'Rata-rata Keseluruhan (μg/m³)': hourly_avg.values.round(2),
    'Hari Kerja (μg/m³)': hourly_avg_weekday.values.round(2),
    'Akhir Pekan (μg/m³)': hourly_avg_weekend.values.round(2),
    'Selisih (μg/m³)': (hourly_avg_weekday.values - hourly_avg_weekend.values).round(2),
    'Kategori': ['Baik' if v <= 15 else 'Sedang' if v <= 35 else 'Tidak Sehat' 
                  for v in hourly_avg.values]
})

print("=" * 90)
print("TABEL POLA HARIAN PM2.5 (μg/m³)")
print("=" * 90)
print(pola_table.to_string(index=False))
print("=" * 90)

# Identifikasi jam dengan PM2.5 tertinggi
max_hour = hourly_avg.idxmax()
min_hour = hourly_avg.idxmin()
print(f"\nJam dengan PM2.5 tertinggi: {max_hour:02d}:00 ({hourly_avg[max_hour]:.2f} μg/m³)")
print(f"Jam dengan PM2.5 terendah: {min_hour:02d}:00 ({hourly_avg[min_hour]:.2f} μg/m³)")

### 3.3 Heatmap Pola Harian

In [ ]:
# Buat heatmap: jam (baris) x hari dalam seminggu (kolom)
df['day_name'] = df['timestamp'].dt.day_name()

# Pivot table untuk heatmap
heatmap_data = df.pivot_table(
    values='pm25', 
    index='hour', 
    columns='day_name',
    aggfunc='mean'
)

# Urutkan kolom sesuai urutan hari
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data[day_order]

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(heatmap_data.values, cmap='RdYlGn_r', aspect='auto')

# Colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('PM2.5 (μg/m³)', fontsize=11)

# Set ticks
ax.set_xticks(range(7))
ax.set_xticklabels(['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu'])
ax.set_yticks(range(24))
ax.set_yticklabels([f'{h:02d}:00' for h in range(24)])

# Labels
ax.set_xlabel('Hari dalam Minggu', fontsize=12)
ax.set_ylabel('Jam (WIB)', fontsize=12)
ax.set_title('Heatmap Pola Harian PM2.5\n(Hari dalam Minggu × Jam)', fontsize=14, fontweight='bold')

# Annotasi nilai di setiap sel
for i in range(24):
    for j in range(7):
        value = heatmap_data.values[i, j]
        text_color = 'white' if value > 30 else 'black'
        ax.text(j, i, f'{value:.0f}', ha='center', va='center', 
               fontsize=8, color=text_color)

plt.tight_layout()
plt.savefig('heatmap_pola_harian_pm25.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nGambar disimpan: heatmap_pola_harian_pm25.png")

## 4. Analisis Jam Sibuk

Analisis khusus untuk jam sibuk pagi (06:00-09:00) dan sore (17:00-20:00).

In [ ]:
# Definisi periode
peak_morning = df[(df['hour'] >= 6) & (df['hour'] < 9)]['pm25']
peak_evening = df[(df['hour'] >= 17) & (df['hour'] < 20)]['pm25']
off_peak = df[(df['hour'] >= 10) & (df['hour'] < 16)]['pm25']
night = df[(df['hour'] >= 0) & (df['hour'] < 6)]['pm25']

print("=" * 70)
print("ANALISIS KONSENTRASI PM2.5 BERDASARKAN PERIODE WAKTU")
print("=" * 70)

print(f"\n{'Periode':<25} {'Rata-rata':>12} {'Std Dev':>10} {'Min':>8} {'Max':>8}")
print("-" * 70)
print(f"{'Malam (00:00-06:00)':<25} {night.mean():>12.2f} {night.std():>10.2f} {night.min():>8.2f} {night.max():>8.2f}")
print(f"{'Pagi (06:00-09:00)':<25} {peak_morning.mean():>12.2f} {peak_morning.std():>10.2f} {peak_morning.min():>8.2f} {peak_morning.max():>8.2f}")
print(f"{'Siang (10:00-16:00)':<25} {off_peak.mean():>12.2f} {off_peak.std():>10.2f} {off_peak.min():>8.2f} {off_peak.max():>8.2f}")
print(f"{'Sore (17:00-20:00)':<25} {peak_evening.mean():>12.2f} {peak_evening.std():>10.2f} {peak_evening.min():>8.2f} {peak_evening.max():>8.2f}")

# Hitung peningkatan
peningkatan_pagi = ((peak_morning.mean() - night.mean()) / night.mean()) * 100
peningkatan_sore = ((peak_evening.mean() - off_peak.mean()) / off_peak.mean()) * 100

print("\n" + "=" * 70)
print("INTERPRETASI")
print("=" * 70)
print(f"\n• Peningkatan PM2.5 dari Malam ke Pagi: +{peningkatan_pagi:.1f}%")
print(f"• Peningkatan PM2.5 dari Siang ke Sore: +{peningkatan_sore:.1f}%")

if peningkatan_pagi > 30:
    print("\n→ Jam sibuk pagi menunjukkan lonjakan signifikan (>{30}%) yang mengindikasikan")
    print("  pengaruh aktivitas通勤 pagi terhadap kualitas udara.")

if peningkatan_sore > 20:
    print("\n→ Jam sibuk sore juga menunjukkan peningkatan yang cukup signifikan")
    print("  mengindikasikan pengaruh aktivitas lalu lintas sore.")

### 4.1 Visualisasi Box Plot Jam Sibuk

In [ ]:
# Siapkan data untuk boxplot
df['periode'] = 'Lainnya'
df.loc[(df['hour'] >= 6) & (df['hour'] < 9), 'periode'] = 'Pagi (06-09)'
df.loc[(df['hour'] >= 17) & (df['hour'] < 20), 'periode'] = 'Sore (17-20)'
df.loc[(df['hour'] >= 10) & (df['hour'] < 16), 'periode'] = 'Siang (10-16)'
df.loc[(df['hour'] >= 0) & (df['hour'] < 6), 'periode'] = 'Malam (00-06)'

# Urutan untuk boxplot
periode_order = ['Malam (00-06)', 'Pagi (06-09)', 'Siang (10-16)', 'Sore (17-20)']

fig, ax = plt.subplots(figsize=(10, 6))

# Buat boxplot
box_data = [df[df['periode'] == p]['pm25'].values for p in periode_order]
bp = ax.boxplot(box_data, labels=periode_order, patch_artist=True, 
                 notch=True, bootstrap=1000)

# Warna untuk setiap box
colors = ['#1e3a5f', '#ef4444', '#22c55e', '#f97316']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Labels dan title
ax.set_xlabel('Periode Waktu', fontsize=12)
ax.set_ylabel('Konsentrasi PM2.5 (μg/m³)', fontsize=12)
ax.set_title('Distribusi PM2.5 pada Berbagai Periode Waktu\n(Box Plot dengan Notch 95% CI)', 
             fontsize=14, fontweight='bold')

# Grid
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Annotasi median
medians = [np.median(d) for d in box_data]
for i, median in enumerate(medians):
    ax.text(i + 1, median + 1, f'{median:.1f}', ha='center', va='bottom', 
            fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('boxplot_periode_waktu.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nGambar disimpan: boxplot_periode_waktu.png")

## 5. Perbandingan Hari Kerja vs Akhir Pekan

Analisis perbedaan konsentrasi PM2.5 antara hari kerja (Senin-Jumat) dan akhir pekan (Sabtu-Minggu).

In [ ]:
# Statistik perbandingan
weekday_pm25 = df[df['is_weekday'] == 1]['pm25']
weekend_pm25 = df[df['is_weekday'] == 0]['pm25']

print("=" * 70)
print("PERBANDINGAN HARI KERJA VS AKHIR PEKAN")
print("=" * 70)

print(f"\n{'Metrik':<30} {'Hari Kerja':>15} {'Akhir Pekan':>15}")
print("-" * 70)
print(f"{'Jumlah Data':<30} {len(weekday_pm25):>15} {len(weekend_pm25):>15}")
print(f"{'Rata-rata PM2.5':<30} {weekday_pm25.mean():>15.2f} {weekend_pm25.mean():>15.2f}")
print(f"{'Median PM2.5':<30} {weekday_pm25.median():>15.2f} {weekend_pm25.median():>15.2f}")
print(f"{'Std Dev':<30} {weekday_pm25.std():>15.2f} {weekend_pm25.std():>15.2f}")
print(f"{'Min':<30} {weekday_pm25.min():>15.2f} {weekend_pm25.min():>15.2f}")
print(f"{'Max':<30} {weekday_pm25.max():>15.2f} {weekend_pm25.max():>15.2f}")

# Selisih
selisih = weekday_pm25.mean() - weekend_pm25.mean()
persen_selisi = (selisih / weekend_pm25.mean()) * 100

print("\n" + "=" * 70)
print("INTERPRETASI")
print("=" * 70)
print(f"\n• Selisih rata-rata: {selisih:.2f} μg/m³")
print(f"• Hari kerja {( 'lebih tinggi' if selisih > 0 else 'lebih rendah')}: {abs(persen_selisi):.1f}%")
print(f"\n• Hari kerja memiliki {abs(persen_selisi):.1f}% PM2.5 {'lebih tinggi' if selisih > 0 else 'lebih rendah'}")
print("  dibanding akhir pekan, mengindikasikan pengaruh aktivitas kerja dan")
print("  lalu lintas通勤 terhadap kualitas udara.")

In [ ]:
# Visualisasi perbandingan hari kerja vs akhir pekan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Line chart perbandingan
ax1 = axes[0]
ax1.plot(hourly_avg_weekday.index, hourly_avg_weekday.values, 
         color='#3b82f6', linewidth=3, marker='o', markersize=8,
         label='Hari Kerja')
ax1.plot(hourly_avg_weekend.index, hourly_avg_weekend.values,
         color='#f59e0b', linewidth=3, marker='s', markersize=8,
         label='Akhir Pekan')
ax1.fill_between(hourly_avg_weekday.index, 
                  hourly_avg_weekday.values, 
                  hourly_avg_weekend.values, 
                  alpha=0.2, color='gray')
ax1.set_xlabel('Jam (WIB)', fontsize=11)
ax1.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax1.set_title('Pola Harian: Hari Kerja vs Akhir Pekan', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left')
ax1.set_xticks(range(0, 24, 2))
ax1.grid(True, alpha=0.3)

# Subplot 2: Bar chart rata-rata
ax2 = axes[1]
categories = ['Hari\nKerja', 'Akhir\nPekan']
means = [weekday_pm25.mean(), weekend_pm25.mean()]
stds = [weekday_pm25.std(), weekend_pm25.std()]

bars = ax2.bar(categories, means, color=['#3b82f6', '#f59e0b'], 
               alpha=0.8, edgecolor='black', linewidth=1.2)
ax2.errorbar(categories, means, yerr=stds, fmt='none', color='black', capsize=5, capthick=2)

# Annotasi
for bar, mean, std in zip(bars, means, stds):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 1, 
            f'{mean:.1f}±{std:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.set_ylabel('Rata-rata PM2.5 (μg/m³)', fontsize=11)
ax2.set_title('Rata-rata PM2.5: Hari Kerja vs Akhir Pekan', fontsize=13, fontweight='bold')
ax2.set_ylim(0, max(means) * 1.4)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('perbandingan_hari_kerja.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nGambar disimpan: perbandingan_hari_kerja.png")

## 6. Kesimpulan dan Interpretasi

Berdasarkan analisis pola harian PM2.5 yang telah dilakukan, dapat ditarik beberapa kesimpulan:

In [ ]:
print("=" * 70)
print("KESIMPULAN ANALISIS POLA HARIAN PM2.5")
print("=" * 70)

print("""
1. POLA HARIAN UMUM
   - PM2.5 menunjukkan pola variasi yang jelas berdasarkan jam
   - Puncak terjadi pada jam sibuk pagi (06:00-09:00) dan sore (17:00-20:00)
   - Nilai terendah pada malam hingga pagi dini (00:00-06:00)

2. JAM SIBUK PAGI
   - Konsentrasi PM2.5 meningkat signifikan saat jam sibuk pagi
   - Didorong oleh aktivitas通勤 dan pemanasan mesin kendaraan
   - Peningkatan ~" + f"{peningkatan_pagi:.0f}% dari periode malam

3. JAM SIBUK SORE
   - Terjadi peningkatan kedua pada sore hari
   - Mencerminkan peak hour lalu lintas dan aktivitas domestik
   - Peningkatan ~" + f"{peningkatan_sore:.0f}% dari periode siang

4. PERBEDAAN HARI KERJA vs AKHIR PEKAN
   - Hari kerja menunjukkan PM2.5 " + (f'lebih tinggi' if selisih > 0 else 'lebih rendah') + " dari akhir pekan
   - Selisih: " + f"{abs(selisih):.2f} μg/m³ ({abs(persen_selisi):.1f}%)
   - Mengindikasikan pengaruh aktivitas kerja dan lalu lintas通勤

5. IMPLIKASI UNTUK SISTEM PREDIKSI
   - Pola harian yang konsisten dapat digunakan sebagai fitur input model
   - Hour-of-day adalah variabel penting untuk prediksi PM2.5
   - Model dapat memanfaatkan pola weekday vs weekend

6. REKOMENDASI UNTUK PENGGUNA
   - Hindari aktivitas luar ruangan pada jam sibuk (07:00-09:00, 17:00-19:00)
   - Waktu terbaik untuk aktivitas luar: pagi dini (00:00-06:00) atau siang (10:00-16:00)
   - Hari weekend relatif memiliki kualitas udara lebih baik
""")

print("=" * 70)

## 7. Export Hasil Analisis

Menyimpan hasil analisis ke file CSV untuk dokumentasi penelitian.

In [ ]:
# Export tabel pola harian
pola_table.to_csv('tabel_pola_harian_pm25.csv', index=False)
print("Tabel pola harian disimpan: tabel_pola_harian_pm25.csv")

# Export statistik per periode
stats_periode = pd.DataFrame({
    'Periode': ['Malam (00:00-06:00)', 'Pagi (06:00-09:00)', 
                'Siang (10:00-16:00)', 'Sore (17:00-20:00)'],
    'Rata-rata': [night.mean(), peak_morning.mean(), off_peak.mean(), peak_evening.mean()],
    'Std_Dev': [night.std(), peak_morning.std(), off_peak.std(), peak_evening.std()],
    'Min': [night.min(), peak_morning.min(), off_peak.min(), peak_evening.min()],
    'Max': [night.max(), peak_morning.max(), off_peak.max(), peak_evening.max()]
})
stats_periode.to_csv('statistik_per_periode.csv', index=False)
print("Statistik per periode disimpan: statistik_per_periode.csv")

# Export heatmap data
heatmap_data.to_csv('heatmap_data_pm25.csv')
print("Data heatmap disimpan: heatmap_data_pm25.csv")

print("\n" + "=" * 50)
print("SEMUA FILE BERHASIL DISIMPAN")
print("=" * 50)
print("\nFile yang dihasilkan:")
print("  - pola_harian_pm25.png (Line chart utama)")
print("  - heatmap_pola_harian_pm25.png (Heatmap)")
print("  - boxplot_periode_waktu.png (Box plot)")
print("  - perbandingan_hari_kerja.png (Perbandingan)")
print("  - tabel_pola_harian_pm25.csv (Data tabel)")
print("  - statistik_per_periode.csv (Statistik)")
print("  - heatmap_data_pm25.csv (Data heatmap)")